In [ ]:
# Code for using SINDy for inferring the equation
# Note, uses separate environment, see environment_sindy.yml

import pysindy as ps
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as spi

In [ ]:
# Load dataset

fpath = "/home/maia-user/Documents/MRE_data/bioqic" # Path to data folder

def loadData(path):
    fname = "four_target_phantom.mat"

    if not path.endswith("/"):
        path += "/"

    fpath = path + fname

    disp = spi.loadmat(fpath)["u_ft"]
    disp = np.transpose(disp, (4, 1, 0, 2, 3)).astype(np.complex128)

    return disp

# Load
disp = loadData(fpath)

In [ ]:
# Prepare dataset
# Data is in Fourier domain, SINDy operates in time domain.
# MRE typically uses eight points in time. Displacements are real valued -> Use RFFT to force +/- symmetry -> 5 points in time.

t_points = 5
t_shape = (*disp.shape,) + (t_points,)
t_disp = np.zeros(t_shape, dtype = disp.dtype)
t_disp[..., 1] = disp
t_disp = np.fft.irfft(t_disp, axis = -1)
t_disp = np.transpose(t_disp, (0,1,2,3,5,4)) # Put time at -2

In [ ]:
# Spatial information

frequencies = np.array([50,60,70,80,90,100])
spacing_all_freqs = []

dx, dy, dz = 0.001, 0.001, 0.001

for i in frequencies:
    dt = 1 / (i * 8)
    spacing = np.array([dx, dy, dz, dt])
    spacing_mesh = np.indices(t_disp.shape[1:-1]).transpose((1,2,3,4,0)) * spacing
    spacing_all_freqs.append(spacing_mesh)

In [ ]:
# Define manual FD differentiation

def np_deriv(data, spacing, dims):
    dx = np.gradient(data, *spacing, axis = dims)
    dx = np.stack(dx, axis = -1)
    return dx

In [ ]:
# Fixed approach
# Here all the derivatives are calculated explicitly and stored for use with PySINDy, including no cross-talk between dimensions other than predefined

freq_idx = 0

data = t_disp[freq_idx, ...]

dx = np_deriv(data, spacing = spacing[:-1], dims = (0,1,2))
dx2 = np_deriv(dx, spacing = spacing[:-1], dims = (0,1,2))

sel = [0,1,2]
div = np.sum(dx[..., sel, sel], axis = -1)

ctemp = [dx[..., 2, 1] - dx[..., 1, 2], dx[..., 0, 2] - dx[..., 2, 0], dx[..., 1, 0] - dx[..., 0, 1]]
curl = np.stack(ctemp, axis = -1)

gdiv = np_deriv(div, spacing[:-1], dims = (0,1,2))
lapl = dx2[..., 0,0] + dx2[..., 1, 1] + dx2[..., 2,2]

bias = np.ones_like(lapl, dtype = lapl.dtype)

lib = [dx[..., 0], dx[..., 1], dx[..., 2], bias * div[..., None], curl, gdiv, lapl, bias]
flib = []
fname = ["dx", "dy", "dz", "div", "curl", "gdiv", "lapl", "bias"]

for l in lib:
    flib.append(np.moveaxis(l, -1, 3))

flib = np.stack(flib, axis = -1)

# Target

ddt_disp = ps.differentiation.FiniteDifference(order = 2, 
                                                d = 2, 
                                                axis = -2, 
                                                periodic=True)._differentiate(t_disp, 1 / (frequencies[freq_idx] * 8))

In [ ]:
# Setup and run regression

ilib = ps.IdentityLibrary().fit(flib)
dtarget = ddt_disp[freq_idx, ...]
dtarget = np.moveaxis(dtarget, -1, 3)[..., None]

opt = ps.STLSQ(threshold = 0.1,
               verbose=True,
               alpha = 0.05)

model = ps.SINDy(opt,
                 ilib,
                 feature_names = fname)

model.fit(flib, x_dot = dtarget, t = spacing_all_freqs[freq_idx][0,0,0,:,3],
          library_ensemble = False)

In [ ]:
# Print results

pred = model.predict(flib).reshape(flib.shape[:-1])
model.equations()

In [ ]:
# Approach with spatial variation

# Basis generation
freq_idx = 0
data = t_disp[freq_idx, ...]

dx = np_deriv(data, spacing = spacing[:-1], dims = (0,1,2))
dx2 = np_deriv(dx, spacing = spacing[:-1], dims = (0,1,2))

sel = [0,1,2]
div = np.sum(dx[..., sel, sel], axis = -1)

ctemp = [dx[..., 2, 1] - dx[..., 1, 2], dx[..., 0, 2] - dx[..., 2, 0], dx[..., 1, 0] - dx[..., 0, 1]]
curl = np.stack(ctemp, axis = -1)

gdiv = np_deriv(div, spacing[:-1], dims = (0,1,2))
lapl = dx2[..., 0,0] + dx2[..., 1, 1] + dx2[..., 2,2]

bias = np.ones_like(lapl, dtype = lapl.dtype)

lib = [dx[..., 0], dx[..., 1], dx[..., 2], bias * div[..., None], curl, gdiv, lapl, bias]
flib = []
fname = ["dx", "dy", "dz", "div", "curl", "gdiv", "lapl", "bias"]

for l in lib:
    flib.append(np.moveaxis(l, -1, 3))

flib = np.stack(flib, axis = -1)

# Target

ddt_disp = ps.differentiation.FiniteDifference(order = 2, 
                                                d = 2, 
                                                axis = -2, 
                                                periodic=True)._differentiate(t_disp, 1 / (frequencies[freq_idx] * 8))

# Create position array
full_pos = np.indices(data.shape[:-2])
full_pos = np.moveaxis(full_pos, 0, -1) * spacing[:-1]

# Fundamental encoding frequncy connected to space
max_freq_idx = 2
sfi = [2 * np.pi * np.arange(nm) / (nm * s) for (nm, s) in zip(data.shape[:3], spacing[:3])]
#sfreqs = [np.fft.fftfreq(full_pos.shape[i], d = spacing[i]) for i in range(3)]
sfreqs = np.stack(np.meshgrid(*sfi, indexing="ij"), axis = -1)
sfreqs = sfreqs[:max_freq_idx, :max_freq_idx, :max_freq_idx, :] # All dirs equal
sfreqs = sfreqs.reshape(-1, sfreqs.shape[-1])

# Explanation: We generate the frequencies of an Fourier series using the definition of the Fourier series. This is then reshaped into a feature vector of max_freq_idx. 
# The first index in sfreqs are the features, the second index correspond to spatial variations.

# Generate full grid
pf_grid = np.einsum("abcd,id->abci", full_pos, sfreqs) # Dot product across the directional terms. 
pf_cos = np.cos(pf_grid) # Both cos and sin required for full coverage
pf_sin = np.sin(pf_grid) # --||--
pf_cs = np.concatenate([pf_cos, pf_sin], axis = -1)

# Now this combines with the standard library generated to form the new one
full_lib = flib[..., :, None] * pf_cs[..., None, None, None, :] # Expand features
full_lib = full_lib.reshape(*full_lib.shape[:-2], full_lib.shape[-2] * full_lib.shape[-1])
pos_enc_lib = ps.IdentityLibrary().fit(full_lib)

# To generate labels
feature_names = fname
modifier_names = ["c_" + ",".join(map(str, row)) for row in sfreqs]
modifier_names += ["s_" + ",".join(map(str, row)) for row in sfreqs]
full_features = [f"{f}_{m}" for f in feature_names for m in modifier_names]

In [ ]:
# Run regression

dtarget = ddt_disp[freq_idx, ...]
dtarget = np.moveaxis(dtarget, -1, 3)[..., None]

opt = ps.STLSQ(threshold = 0.1,
               verbose=True,
               alpha = 0.05)

model2 = ps.SINDy(opt,
                 pos_enc_lib,
                 feature_names = full_features)

model2.fit(full_lib, x_dot = dtarget, t = spacing_all_freqs[freq_idx][0,0,0,:,3],
          library_ensemble = False)

In [ ]:
# Reshape

pred2 = model2.predict(full_lib).reshape(full_lib.shape[:-1])

In [ ]:
# Print both results

d, t, z = 2, 0, 5

vmax = np.amax(dtarget[:,:,z,d,t,0])
fig, axs = plt.subplots(1, 3)
fig.set_size_inches(11,3.6)

axs[0].imshow(pred[:,:,z,d,t].transpose(), vmax = vmax)
axs[0].set_axis_off()
axs[0].set_title("No spatial variation")

axs[1].imshow(pred2[:,:,z,d,t].transpose(), vmax = vmax)
axs[1].set_axis_off()
axs[1].set_title("2 Spatial frequencies")

im = axs[2].imshow(dtarget[:,:,z,d,t,0].transpose(), vmax = vmax)
axs[2].set_axis_off()
axs[2].set_title("Target")
fig.subplots_adjust(right = 0.92)
cbar_ax = fig.add_axes([0.94,0.125,0.01,0.75])
plt.colorbar(im, cbar_ax)

# fig.savefig("sindy_pred.png", transparent=True,
#             dpi = 600,
#             format = "png",
#             bbox_inches="tight")